# COGS 109 | Step 0: Build the NBA player dataset
**Project:** Built for the NBA? Can pre-draft physical attributes predict early-career scoring?

**Outcome:** For a prospect attending the NBA Combine in year *t*, sum NBA regular-season points across seasons *t-(t+1)*, *(t+1)-(t+2)*, and *(t+2)-(t+3)*, and divide by the sum of games played. E.g., a 2018 combine participant is evaluated using 2018-19, 2019-20, and 2020-21. Only seasons fully completed by 2024-25 are used, which allows combine years **2001–2022**.

The output includes only players who appeared in at least one NBA game in their three-year window. Prospects with no games are **not** assigned zero PPG. We save those players separately in an audit file.

**Sources:** [Official NBA Draft Combine endpoint](https://github.com/swar/nba_api/blob/master/docs/nba_api/stats/endpoints/draftcombinestats.md) and [official league player totals endpoint](https://github.com/swar/nba_api/blob/master/docs/nba_api/stats/endpoints/leaguedashplayerstats.md). The `nba_api` Python package accesses these endpoints. The downloaded raw responses are cached locally.

**Important:** This notebook downloads *real data* when it runs on a computer with Internet access. Do not use the synthetic testing fixture as project results.

## 1. Setup
Keep this notebook and `nba_cp1_data.py` in the same folder. Run from that folder. The NBA API can rate-limit or block some environments. If it fails, obtain complete source CSVs and place them in `raw/combine.csv` and `raw/season_stats.csv`, then rerun. Required raw columns are described in the README.

In [ ]:
%pip -q install pandas numpy matplotlib seaborn statsmodels scikit-learn nba_api

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display
assert Path('nba_cp1_data.py').exists(), 'Put nba_cp1_data.py in the notebook working directory.'
from nba_cp1_data import build_from_cached_or_api
print('Project folder:', Path.cwd())

## 2. Download and assemble
The script downloads 2001–2024 season totals and combine measurements, matches records by **NBA player ID**, and calculates games-weighted PPG. It stops if a complete season is unavailable rather than incorrectly labeling those games as zero.

In [ ]:
players, audit = build_from_cached_or_api()

## 3. Validate the data
Do not skip these checks. Also review the audit CSV to quantify who is missing from the modeling sample.

In [ ]:
display(players.head(8))
print('Players in model sample:', len(players))
print('Unique IDs:', players.player_id.nunique())
print('Combine years:', players.combine_year.min(), players.combine_year.max())
print('PPG mean / median:', round(players.ppg_3yr.mean(),2), round(players.ppg_3yr.median(),2))
print('Position distribution:')
print(players.position_group.value_counts().to_string())
assert len(players) == players.player_id.nunique()
assert players.ppg_3yr.notna().all()
assert (players.total_games_3yr > 0).all()
assert (players.ppg_3yr >= 0).all()

In [ ]:
missing = (100 * players.isna().mean()).sort_values(ascending=False).round(1)
display(missing.to_frame('Missing (%)'))
print('Full combine participants:', len(audit))
print('NBA appearance in first 3 seasons:', int(audit.made_nba_within_3yr.sum()))
print('No NBA appearance in first 3 seasons:', int((audit.made_nba_within_3yr==0).sum()))
print('Rows available for PPG:',len(players))

## 4. View the Checkpoint 1 figures
These figures are generated when a member runs a model notebook. Each member must separately inspect assumption diagnostics.

In [ ]:
from pathlib import Path
print('Modeling CSV:', (Path('output')/'nba_early_career_ppg.csv').resolve())
print('Audit:', (Path('output')/'combine_join_audit.csv').resolve())
print('All group members must use this exact generated CSV for their models.')

### Notes for your presentation
- State the number of eligible participants **after** joining records and excluding non-NBA appearances.
- List the numeric predictor count and one qualitative predictor (position).
- Note that players with missing athletic tests are imputed from **training data only** later, and their missingness itself may be non-random.
- If coverage is unexpectedly low, investigate missing seasons, combine coverage, and positions **before** modeling.